In [1]:
print("hii")

hii


In [2]:
import os
import json
from pathlib import Path

print("✅ Imports ready")

✅ Imports ready


In [25]:
import os
import json


def load_all_json(root_folder, data_folder):

    folder_path = os.path.join(root_folder, data_folder)

    all_docs = []

    for root, dirs, files in os.walk(folder_path):

        for file in files:
            if file.endswith(".json"):

                file_path = os.path.join(root, file)

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)

                        if isinstance(data, list):
                            all_docs.extend(data)
                        else:
                            print("⚠ Not list:", file_path)

                except Exception as e:
                    print("❌ Error:", file_path, e)

    return all_docs


# Root folder
ROOT_FOLDER =  os.getcwd()

# Data folder inside root
DATA_FOLDER = r"..\CyberLawsData"


all_docs = load_all_json(ROOT_FOLDER, DATA_FOLDER)

print("Total docs:", len(all_docs))

Total docs: 25


In [4]:
REQUIRED_FIELDS = [
    "id",
    "title",
    "url",
    "category",
    "tags",
    "language",
    "date",
    "content"
]


def validate_docs(docs):

    valid = []
    invalid = []

    for d in docs:

        ok = True

        for f in REQUIRED_FIELDS:
            if f not in d:
                ok = False

        if ok:
            valid.append(d)
        else:
            invalid.append(d)

    return valid, invalid


valid_docs, invalid_docs = validate_docs(all_docs)

print("Valid:", len(valid_docs))
print("Invalid:", len(invalid_docs))

Valid: 25
Invalid: 0


In [5]:
for i, d in enumerate(valid_docs[:5]):

    print("\n------", i, "------")

    print("ID:", d["id"])
    print("Title:", d["title"])
    print("Tags:", d["tags"])
    print("Category:", d["category"])
    print("Content:", d["content"][:150])


------ 0 ------
ID: doc_001
Title: IT Act Section 65 – Tampering with computer source documents
Tags: ['section65', 'source_code', 'tampering', 'cybercrime', 'india']
Category: cyber_law
Content: Section 65 – Tampering with computer source documents. A person who intentionally conceals, destroys or alters any computer source code such as progra

------ 1 ------
ID: doc_002
Title: IT Act Section 66 – Using password of another person
Tags: ['section66', 'password', 'digital_signature', 'fraud', 'cybercrime', 'india']
Category: cyber_law
Content: Section 66 – Using password of another person. If a person fraudulently uses the password, digital signature or other unique identification of another

------ 2 ------
ID: doc_003
Title: IT Act Section 66D – Cheating using computer resource
Tags: ['section66D', 'cheating', 'fraud', 'computer_resource', 'cybercrime', 'india']
Category: cyber_law
Content: Section 66D – Cheating using computer resource. If a person cheats someone using a computer r

In [6]:
from langchain_core.documents import Document

print("✅ Document import ready")

✅ Document import ready


In [7]:
def build_cyberlaw_documents(docs):

    documents = []

    for d in docs:

        title = d.get("title", "")
        content = d.get("content", "")
        tags = d.get("tags", [])
        url = d.get("url", "")
        date = d.get("date", "")
        category = d.get("category", "")

        tags_text = ", ".join(tags)

        text = f"""
Title: {title}

Category: {category}

Tags: {tags_text}

Date: {date}

Content:
{content}
"""

        metadata = {
            "id": d.get("id"),
            "title": title,
            "tags": tags,
            "url": url,
            "date": date,
            "category": category
        }

        documents.append(
            Document(
                page_content=text.strip(),
                metadata=metadata
            )
        )

    return documents

In [9]:
documents = build_cyberlaw_documents(valid_docs)

print("Total documents:", len(documents))

Total documents: 25


In [10]:
for i in range(min(3, len(documents))):

    print("\n------ DOC ------")
    print(documents[i].page_content)
    print("\nMetadata:")
    print(documents[i].metadata)


------ DOC ------
Title: IT Act Section 65 – Tampering with computer source documents

Category: cyber_law

Tags: section65, source_code, tampering, cybercrime, india

Date: 2020-01-01

Content:
Section 65 – Tampering with computer source documents. A person who intentionally conceals, destroys or alters any computer source code such as programmes, computer commands, design and layout, when it is required to be maintained by law, commits an offence and can be punished with imprisonment up to 3 years or fine up to 2 lakhs INR or both.

Metadata:
{'id': 'doc_001', 'title': 'IT Act Section 65 – Tampering with computer source documents', 'tags': ['section65', 'source_code', 'tampering', 'cybercrime', 'india'], 'url': 'https://infosecawareness.in/cyber-laws-of-india', 'date': '2020-01-01', 'category': 'cyber_law'}

------ DOC ------
Title: IT Act Section 66 – Using password of another person

Category: cyber_law

Tags: section66, password, digital_signature, fraud, cybercrime, india

Date:

In [14]:
import os
from dotenv import load_dotenv

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("✅ Imports ready")

✅ Imports ready


In [15]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY missing")

print("✅ API key loaded")

✅ API key loaded


In [16]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

print("✅ Embeddings ready")

✅ Embeddings ready


In [2]:
import os
DB_PATH =  os.getcwd() + r"\..\RAG\faiss_cyberlaw_index_india"

print("DB path:", DB_PATH)

DB path: h:\PGAGI\CyberGuard_Project\cyberGuard\test_Scripts\..\RAG\faiss_cyberlaw_index_india


In [ ]:
if os.path.exists(DB_PATH):

    print("📂 Loading existing DB")

    db = FAISS.load_local(
        DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )

    print("➕ Adding documents")

    db.add_documents(documents)

else:

    print("🆕 Creating new DB")

    db = FAISS.from_documents(
        documents,
        embeddings
    )


db.save_local(DB_PATH)

print("✅ DB ready")
print("Total docs:", len(db.index_to_docstore_id))

🆕 Creating new DB
✅ DB ready
Total docs: 25


In [27]:
def retrieve(query, k=4):

    results = db.similarity_search_with_score(query, k=k)

    for i, (doc, score) in enumerate(results):

        print("\n------", i+1, "------")
        print("Score:", score)
        print(doc.page_content[:300])


retrieve("data protection law")


------ 1 ------
Score: 0.56721735
Title: Digital Personal Data Protection Act 2023

Category: cyber_law

Tags: data_protection, privacy, personal_data, dpdp, 2023

Date: 2023-08-11

Content:
The Digital Personal Data Protection Act, 2023. An Act to provide for processing of digital personal data recognizing the right of individuals 

------ 2 ------
Score: 0.6419877
Title: IT Act Section 43A – Data protection at corporate level

Category: cyber_law

Tags: section43A, data_protection, corporate, security, cyber_law, india

Date: 2020-01-01

Content:
Section 43A – Data protection at corporate level. If a body corporate fails to implement reasonable security pract

------ 3 ------
Score: 0.67608064
Title: Information Technology Act 2000 Cyber Law

Category: cyber_law

Tags: it_act, cyberlaw, electronic_records, digital_signature, 2000

Date: 2000-06-09

Content:
The Information Technology Act, 2000. An Act to provide legal recognition for electronic commerce, electronic records and digit

## Retrival

In [4]:
import os
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

# Load env variables
load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY missing")

print("✅ API key loaded")

# Initialize embeddings (same as used during creation)
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

# Path to saved FAISS DB
DB_PATH = os.getcwd() + r"\..\RAG\faiss_cyberlaw_index_india"

# Load FAISS index
db = FAISS.load_local(
    DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print("✅ FAISS DB loaded")


# 🔍 Retrieve function
def retrieve_metadata(query, k=4):
    results = db.similarity_search_with_score(query, k=k)

    for i, (doc, score) in enumerate(results):
        print(f"\n------ Result {i+1} ------")
        print("Score:", score)

        # 👉 Metadata only
        print("ID:", doc.metadata.get("id"))
        print("Title:", doc.metadata.get("title"))
        print("Category:", doc.metadata.get("category"))
        print("Tags:", doc.metadata.get("tags"))
        print("URL:", doc.metadata.get("url"))
        print("Date:", doc.metadata.get("date"))


# Example query
retrieve_metadata("data protection law")

✅ API key loaded
✅ FAISS DB loaded


GoogleGenerativeAIError: Error embedding content (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}